# PySpark reference 1: execution model

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-01-execution-model.ipynb)

Sessions and configuration, lazy evaluation, shuffles, jobs, stages and tasks, partitions, reading physical plans, pushdown, AQE, caching and checkpointing, each with a runnable example. Practise it in [PySpark practice 1: execution model](pyspark-practice-01-execution-model.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Sessions and configuration](#sessions-and-configuration) | [`SparkSession.builder`](#sparksessionbuilder), [`spark.conf`](#sparkconf) |
| [Lazy evaluation](#lazy-evaluation) | [Transformations and actions](#transformations-and-actions), [Narrow and wide transformations](#narrow-and-wide-transformations) |
| [Jobs, stages and tasks](#jobs-stages-and-tasks) | [How an action becomes jobs, stages and tasks](#how-an-action-becomes-jobs-stages-and-tasks) |
| [Partitions](#partitions) | [Partition count: reading, shuffling and AQE](#partition-count-reading-shuffling-and-aqe), [`repartition` and `coalesce`](#repartition-and-coalesce) |
| [Reading query plans](#reading-query-plans) | [`DataFrame.explain(mode=None)`](#dataframeexplainmodenone), [Physical operators you will see](#physical-operators-you-will-see), [Predicate and projection pushdown](#predicate-and-projection-pushdown), [Adaptive Query Execution (AQE)](#adaptive-query-execution-aqe) |
| [Caching and checkpointing](#caching-and-checkpointing) | [`cache`, `persist` and `unpersist`](#cache-persist-and-unpersist), [`checkpoint` and `localCheckpoint`](#checkpoint-and-localcheckpoint), [Broadcast variables and accumulators](#broadcast-variables-and-accumulators) |

## Sessions and configuration

### `SparkSession.builder`

The entry point to the DataFrame and SQL APIs. `builder` collects settings and `getOrCreate()` returns the active
session if one exists in the process, otherwise creates one. A session owns one `SparkContext` (the connection to
the cluster), the catalog, and the runtime SQL configuration.

| method | description |
| --- | --- |
| `.master(url)` | where to run: `local[N]` (N threads), `local[*]` (all cores), `yarn`, `k8s://...`, `spark://host:7077`. Usually set by `spark-submit` or the platform, not in code |
| `.appName(name)` | the name shown in the Spark UI and cluster manager |
| `.config(key, value)` | any Spark setting; static settings (memory, cores, extensions) only take effect before the session starts |
| `.remote("sc://host:15002")` | connect through Spark Connect instead of running the driver locally |
| `.enableHiveSupport()` | use a Hive metastore as the catalog |
| `.getOrCreate()` | return the active session or build a new one |

**Returns:** a `SparkSession`.

**Notes**
- `getOrCreate()` silently **reuses** an existing session and ignores static settings passed afterwards; restart
  the process (or `spark.stop()`) to change memory or cores.
- On Databricks, EMR, Dataproc and Glue the platform creates `spark` for you; build your own only in scripts and tests.

In [ ]:
same = SparkSession.builder.appName("ignored").getOrCreate()
print("same session object:", same is spark)
print("version:", spark.version, "| master:", spark.sparkContext.master, "| app:", spark.sparkContext.appName)

### `spark.conf`

The runtime configuration of the session. Read with `get`, change SQL settings with `set`, restore a default with
`unset`. Settings changed at runtime apply to queries planned afterwards.

| method | description |
| --- | --- |
| `spark.conf.get(key, default=None)` | current value (raises if unset and no default) |
| `spark.conf.set(key, value)` | change a runtime setting |
| `spark.conf.unset(key)` | back to the default |
| `spark.conf.isModifiable(key)` | whether it can change after startup |
| `spark.sparkContext.getConf().getAll()` | the static settings the application started with |

**Settings worth knowing**

| setting | default | controls |
| --- | --- | --- |
| `spark.sql.shuffle.partitions` | 200 | partitions after a shuffle (joins, aggregations, windows) |
| `spark.sql.adaptive.enabled` | true | Adaptive Query Execution |
| `spark.sql.autoBroadcastJoinThreshold` | 10MB | automatic broadcast joins |
| `spark.sql.files.maxPartitionBytes` | 128MB | maximum bytes per partition when reading files |
| `spark.sql.session.timeZone` | JVM zone | how timestamps are shown and dates derived |
| `spark.sql.ansi.enabled` | true (Spark 4) | errors instead of silent nulls on bad casts and overflow |

**Notes:** setting a static key such as `spark.executor.memory` with `spark.conf.set` raises
`CANNOT_MODIFY_CONFIG` (or has no effect); pass it to the builder or `spark-submit`.

In [ ]:
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("after set:         ", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("modifiable? shuffle partitions:", spark.conf.isModifiable("spark.sql.shuffle.partitions"),
      "| executor memory:", spark.conf.isModifiable("spark.executor.memory"))

## Lazy evaluation

### Transformations and actions

A **transformation** (`select`, `where`, `join`, `groupBy().agg`, `withColumn`, `orderBy`, ...) returns a new
DataFrame describing a computation; nothing runs. An **action** (`count`, `collect`, `show`, `take`, `first`,
`toPandas`, `write...save`, `foreach`) asks for a result, so Spark optimizes the whole chain of transformations
into one plan and runs it.

| actions | what they return |
| --- | --- |
| `show(n)`, `take(n)`, `head(n)`, `first()`, `limit(n).collect()` | a few rows (stop early when possible) |
| `count()` | a row count |
| `collect()`, `toPandas()`, `toLocalIterator()` | every row, on the driver |
| `df.write...save()`, `saveAsTable`, `insertInto` | nothing; writes data |
| `foreach`, `foreachPartition` | nothing; runs a function on executors |

**Notes**
- Errors in a transformation, such as a bad cast under ANSI mode, surface only when an action runs, often far
  from the line that caused them, and only if the action actually **needs** that column: the optimizer prunes
  unused columns, so `count()` can succeed on a DataFrame whose `collect()` fails.
- Every action recomputes the plan from the source unless an intermediate result is cached. Two actions on the same
  DataFrame read the input twice.
- Analysis errors (unknown column, wrong type) are raised **eagerly**, when the transformation is defined.

In [ ]:
lazy = orders.withColumn("amount_int", F.col("status").cast("int"))  # 'paid' cannot become an int, but nothing ran yet
print("transformation defined; columns:", lazy.columns)
print("count() still works, the unused column is pruned:", lazy.count())
with expected_error():
    lazy.collect()  # this action needs amount_int, so the cast runs and fails
with expected_error():
    orders.select("no_such_column")  # analysis errors are raised immediately

### Narrow and wide transformations

A **narrow** transformation computes each output partition from one input partition (`select`, `where`,
`withColumn`, `union`, `coalesce`, map-side work). A **wide** transformation needs rows with the same key from every
input partition, so Spark **shuffles**: each task writes its rows into buckets by key, and the next stage reads its
bucket from every task. Wide operations: `groupBy`, `join` (unless broadcast), `distinct`, `orderBy`, `repartition`,
window functions.

**Notes:** a shuffle is the expensive part of most jobs (disk writes, network, a stage boundary). It shows up in the
physical plan as an `Exchange` operator; counting `Exchange` lines is the quickest way to count shuffles.

In [ ]:
def shuffles(df):
    return sum("Exchange" in line for line in plan_lines(df))

print("select + where:      ", shuffles(orders.select("order_id", "amount").where("amount > 100")))
print("groupBy:             ", shuffles(orders.groupBy("status").count()))
print("orderBy:             ", shuffles(orders.orderBy("amount")))
print("groupBy then orderBy:", shuffles(orders.groupBy("status").count().orderBy("count")))

## Jobs, stages and tasks

### How an action becomes jobs, stages and tasks

- An **action** starts one or more **jobs**.
- Each job is split into **stages** at shuffle boundaries: everything between two shuffles runs as one stage,
  pipelined row by row without materializing intermediate results.
- Each stage runs one **task** per partition. A task is the unit of parallelism: with 8 executor cores, 8 tasks run
  at once.

| term | created by | count |
| --- | --- | --- |
| job | an action (sometimes several per action, e.g. sorting samples the data first) | per action |
| stage | a shuffle boundary | shuffles + 1 per job (before AQE merges or skips stages) |
| task | a partition of a stage | partitions of that stage |

**Notes:** the Spark UI (port 4040 by default) shows jobs, stages, task durations, shuffle sizes and spill. A stage
whose tasks have very uneven durations points to skew. `spark.sparkContext.statusTracker()` exposes the same
information programmatically.

In [ ]:
tracker = spark.sparkContext.statusTracker()
spark.sparkContext.setJobGroup("demo", "count paid orders per customer")
orders.where("status = 'paid'").groupBy("customer_id").count().collect()
job_ids = tracker.getJobIdsForGroup("demo")
for job_id in sorted(job_ids):
    info = tracker.getJobInfo(job_id)
    stages = [tracker.getStageInfo(s) for s in info.stageIds]
    print(f"job {job_id}: stages {[ (s.stageId, s.numTasks) for s in stages if s ]}")
spark.sparkContext.setJobGroup("other", "")

## Partitions

### Partition count: reading, shuffling and AQE

A partition is a chunk of rows processed by one task. Where the count comes from:

| situation | partitions |
| --- | --- |
| reading files | files are split into chunks of at most `spark.sql.files.maxPartitionBytes` (128 MB), with small files packed together |
| `spark.createDataFrame(local data)` | `spark.sparkContext.defaultParallelism` (the core count) |
| after a shuffle | `spark.sql.shuffle.partitions` (200), then AQE may **coalesce** small partitions |
| `repartition(n)` / `coalesce(n)` | exactly `n` (coalesce can only reduce) |

| inspect with | returns |
| --- | --- |
| `df.rdd.getNumPartitions()` | the number of partitions |
| `F.spark_partition_id()` | each row's partition id, to measure the spread |

**Notes:** too few partitions leave cores idle and make tasks spill; too many create scheduling overhead and tiny
output files. A common target is partitions of 100–200 MB after a shuffle, and two to four tasks per core.

In [ ]:
print("local data:", orders.rdd.getNumPartitions(), "partitions (defaultParallelism =", spark.sparkContext.defaultParallelism, ")")
path = f"{WORK_DIR}/orders_parquet"
orders.repartition(6).write.mode("overwrite").parquet(path)
print("read 6 files:", spark.read.parquet(path).rdd.getNumPartitions(), "partitions (small files are packed together)")
spark.conf.set("spark.sql.files.maxPartitionBytes", "1k")
spark.conf.set("spark.sql.files.openCostInBytes", "1")
print("maxPartitionBytes=1k:", spark.read.parquet(path).rdd.getNumPartitions(), "partitions")
spark.conf.unset("spark.sql.files.maxPartitionBytes")
spark.conf.unset("spark.sql.files.openCostInBytes")

grouped = orders.groupBy("customer_id").count()
grouped.collect()
print("after groupBy with AQE:", grouped.rdd.getNumPartitions(), "partition(s), from shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

### `repartition` and `coalesce`

| method | shuffle | result |
| --- | --- | --- |
| `df.repartition(n)` | yes | `n` partitions of roughly equal size (round-robin) |
| `df.repartition(n, *cols)` / `repartition(*cols)` | yes | rows hashed by `cols`: the same key always lands in the same partition |
| `df.repartitionByRange(n, *cols)` | yes | sorted ranges of `cols` (samples the data first) |
| `df.coalesce(n)` | no | merges existing partitions into `n`; cannot increase the count |

**Use them for:** `repartition(col)` before writing partitioned output or before several joins on the same key;
`coalesce(n)` to reduce the number of output files cheaply at the end of a job.

**Notes:** `coalesce` reduces parallelism of the **whole stage** it ends, not just the write: `coalesce(1)` after
an expensive transformation makes that transformation run in one task. Use `repartition(1)` if the upstream work
must stay parallel. Performance tuning is covered in [PySpark reference 13: performance and optimization](pyspark-reference-13-performance.ipynb).

In [ ]:
wide = spark.range(1000, numPartitions=8)
print("repartition(4):", wide.repartition(4).rdd.getNumPartitions(), "| coalesce(2):", wide.coalesce(2).rdd.getNumPartitions(),
      "| coalesce(16):", wide.coalesce(16).rdd.getNumPartitions())
by_customer = orders.repartition(3, "customer_id").select("customer_id", F.spark_partition_id().alias("partition"))
by_customer.groupBy("partition").agg(F.sort_array(F.collect_set("customer_id")).alias("customers")).orderBy("partition").show()

## Reading query plans

### `DataFrame.explain(mode=None)`

Prints the plan Spark will use for this DataFrame. Before an action it shows the plan built from estimates; after an
action, with AQE, it shows the final plan and the initial plan.

| `mode` | shows |
| --- | --- |
| `"simple"` (default, or `explain()`) | the physical plan |
| `"extended"` | parsed, analyzed and optimized logical plans, plus the physical plan |
| `"formatted"` | a numbered operator tree followed by each operator's details (filters, pushed predicates, output columns) |
| `"cost"` | the optimized logical plan with size and row estimates (`sizeInBytes`, `rowCount`) |
| `"codegen"` | the generated Java code for whole-stage code generation |

**Returns:** `None` (prints). Capture it with `contextlib.redirect_stdout`, as `plan_lines(df)` does.

**How to read it:** the physical plan is a tree read **bottom up**: scans at the leaves, then filters and projections,
then exchanges, aggregates, joins and sorts. Operators prefixed with `*(n)` run inside whole-stage code generation
stage `n`.

In [ ]:
query = orders.where("status = 'paid'").groupBy("customer_id").agg(F.sum("amount").alias("revenue"))
query.explain()
print("\n".join(plan_lines(query, "formatted")[:12]))

### Physical operators you will see

| operator | meaning |
| --- | --- |
| `Scan` / `FileScan parquet` / `LocalTableScan` | reading the source; file scans list `PushedFilters`, `PartitionFilters` and `ReadSchema` |
| `Filter`, `Project` | `where` and `select` |
| `Exchange hashpartitioning(k, n)` | a shuffle by key into `n` partitions |
| `Exchange rangepartitioning(...)` | a shuffle for a global sort |
| `Exchange SinglePartition` | everything to one partition (window or sort with no partitioning) |
| `HashAggregate` / `ObjectHashAggregate` / `SortAggregate` | aggregation; usually a `partial_` step before the exchange and a final step after |
| `Sort` | sorting within partitions |
| `BroadcastExchange` + `BroadcastHashJoin` | a broadcast join |
| `SortMergeJoin`, `ShuffledHashJoin`, `BroadcastNestedLoopJoin`, `CartesianProduct` | other join strategies |
| `Window` | window functions |
| `AdaptiveSparkPlan isFinalPlan=...` | AQE wrapper; `true` once the query has run |
| `InMemoryTableScan` | reading a cached DataFrame |

**Notes:** the two-step aggregate (`partial_sum` → `Exchange` → `sum`) is why `groupBy` is cheaper than it looks:
only one partial row per key and task crosses the network.

### Predicate and projection pushdown

For columnar file formats (Parquet, ORC) and many connectors, Spark pushes `where` conditions and the list of needed
columns into the scan. The source then skips row groups whose min/max statistics cannot match, and reads only the
selected columns. Look for `PushedFilters` and `ReadSchema` in the `FileScan` line; for directories partitioned by a
column, `PartitionFilters` shows which folders were skipped entirely (partition pruning).

**Notes:** a filter wrapped in a UDF, or on an expression the source cannot evaluate, is not pushed down and runs
after the full scan. Casting the column (`cast(col as string) = '5'`) can also block pushdown.

In [ ]:
pq = f"{WORK_DIR}/orders_by_status"
orders.write.mode("overwrite").partitionBy("status").parquet(pq)
scan = spark.read.parquet(pq).where("status = 'paid' AND amount > 100").select("order_id", "amount")
for line in plan_lines(scan, "formatted"):
    if any(k in line for k in ("PushedFilters", "PartitionFilters", "ReadSchema")):
        print(line.strip()[:110])

### Adaptive Query Execution (AQE)

AQE re-optimizes a query **while it runs**, at each shuffle boundary, using the real size of the shuffled data
instead of estimates. It is on by default (`spark.sql.adaptive.enabled`).

| optimization | effect | setting |
| --- | --- | --- |
| coalesce shuffle partitions | merges small post-shuffle partitions | `spark.sql.adaptive.coalescePartitions.enabled` |
| switch join strategy | turns a sort-merge join into a broadcast join when one side turns out small | `spark.sql.adaptive.autoBroadcastJoinThreshold` |
| skew join | splits oversized partitions of a sort-merge join | `spark.sql.adaptive.skewJoin.enabled` |
| local shuffle reader | reads shuffle files locally after a strategy switch | `spark.sql.adaptive.localShuffleReader.enabled` |

**Notes:** `explain()` before an action shows `isFinalPlan=false` and the estimated plan; after an action it shows
the `Final Plan` and the `Initial Plan`. Plans in the Spark UI SQL tab show the final version.

In [ ]:
joined = orders.join(customers, "customer_id").groupBy("region").count()
print("before running:", [l.strip()[:60] for l in plan_lines(joined) if "isFinalPlan" in l or "SortMergeJoin" in l][:2])
print("after running: ", [l.strip()[:60] for l in final_plan(joined).splitlines() if "Join" in l or "AQEShuffleRead" in l][:3])

## Caching and checkpointing

### `cache`, `persist` and `unpersist`

`df.persist(storageLevel)` marks a DataFrame to be kept after it is first computed; later actions read the stored
partitions instead of recomputing the plan. `df.cache()` is `persist()` with the default level, `MEMORY_AND_DISK`
(stored deserialized in memory, spilling to disk when memory is short). Caching is lazy: the data is stored by the
**first action**.

| storage level | where | trade-off |
| --- | --- | --- |
| `MEMORY_AND_DISK` (default) | memory, overflow to disk | the safe default |
| `MEMORY_ONLY` | memory; partitions that do not fit are recomputed | fastest when it fits |
| `DISK_ONLY` | local disk | for large, expensive results |
| `MEMORY_AND_DISK_2`, ... | the same, replicated twice | survives an executor loss |
| `OFF_HEAP` | off-heap memory | needs off-heap memory configured |

| method / property | description |
| --- | --- |
| `df.cache()` / `df.persist(level)` | mark for caching (returns the same DataFrame) |
| `df.unpersist(blocking=False)` | drop the cached data |
| `df.is_cached`, `df.storageLevel` | inspect |
| `spark.catalog.clearCache()` | drop everything cached in the session |

**Use it for:** a DataFrame that is expensive to compute and used by **several actions** (a cleaned table feeding
three outputs, an iterative algorithm).

**Notes:** caching costs memory and a full computation; for a DataFrame used once it only adds work. Unpersist when
done. A cached plan shows `InMemoryTableScan`.

In [ ]:
from pyspark import StorageLevel

enriched = orders.join(customers, "customer_id").where("status <> 'returned'")
enriched.persist(StorageLevel.MEMORY_AND_DISK)
print("is_cached:", enriched.is_cached, "| level:", enriched.storageLevel)
enriched.count()  # first action stores the partitions
print("reads cache:", any("InMemoryTableScan" in l for l in plan_lines(enriched.groupBy("region").count())))
enriched.unpersist()
print("after unpersist:", enriched.is_cached)

### `checkpoint` and `localCheckpoint`

Both materialize a DataFrame and **cut its lineage**: the returned DataFrame's plan starts from the saved data
instead of the original sources. Use them when a plan grows very long (iterative loops that keep adding
transformations), which slows planning and can overflow the stack.

| method | stores in | reliable after an executor loss |
| --- | --- | --- |
| `df.checkpoint(eager=True)` | `spark.sparkContext.setCheckpointDir(path)` (reliable storage) | yes |
| `df.localCheckpoint(eager=True)` | executor storage | no; faster |

**Returns:** a new DataFrame (use the return value; the original keeps its lineage).

In [ ]:
looped = orders.select("order_id", "amount")
for i in range(5):
    looped = looped.withColumn("amount", F.col("amount") + 1)
print("plan lines before:", len(plan_lines(looped, "extended")))
cut = looped.localCheckpoint()
print("plan lines after localCheckpoint:", len(plan_lines(cut, "extended")))
spark.sparkContext.setCheckpointDir(f"{WORK_DIR}/checkpoints")
print("reliable checkpoint rows:", looped.checkpoint().count())

### Broadcast variables and accumulators

Lower-level tools from the RDD API that still appear in older code:

- `sc.broadcast(value)` ships a read-only Python object to every executor once (a lookup dict used inside a UDF).
  With DataFrames, prefer a broadcast **join** to a small DataFrame.
- `sc.accumulator(0)` is a counter executors add to and the driver reads, for example to count bad records inside a
  `foreach`. Updates inside transformations can be applied more than once when tasks are retried, so treat them as
  approximate unless they are updated in actions.

In [ ]:
rates = spark.sparkContext.broadcast({"paid": 1.0, "shipped": 1.0, "placed": 0.5, "returned": 0.0})
weight = F.udf(lambda status: rates.value.get(status, 0.0), "double")
orders.select("status", weight("status").alias("weight")).distinct().orderBy("status").show()

seen = spark.sparkContext.accumulator(0)
orders.foreach(lambda row: seen.add(1))
print("rows counted by the accumulator:", seen.value)

**Next:** practise these entries in [PySpark practice 1: execution model](pyspark-practice-01-execution-model.ipynb).